# Clustering Analysis (K-Means)

This notebook classifies each telemetry window into one of three archetypes (Combat, Collection, Exploration) using K-Means clustering, and then aggregates these into a per-player behavioral profile.

In [ ]:
import pandas as pd
import os
import numpy as np
from sklearn.cluster import KMeans

INPUT_PATH = 'data/processed/4_activity_contributions.csv'
OUT_CLUSTERED = 'data/processed/5_clustered_telemetry.csv'
OUT_ARCHETYPES = 'data/processed/5_player_archetypes.csv'

print("Loading activity data...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df)} rows.")
    
    # Select Features for Clustering
    # We use the percentage contributions for the 3 activities
    features = ['pct_combat', 'pct_collect', 'pct_explore']
    X = df[features].fillna(0)
    
    print("\n--- Running K-Means (K=3) ---")
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df['cluster'] = kmeans.fit_predict(X)

    # Compute Distances to Assigned Cluster Centroid
    # transform(X) returns distance to ALL centroids
    dists = kmeans.transform(X)
    # min(axis=1) gives distance to the closest (assigned) center.
    # Note: This is an inverse measure of archetype strength (Lower distance = Stronger affiliation).
    # Used to derive soft affiliation strength in the ANFIS layer.
    df['distance_to_centroid'] = dists.min(axis=1)
    
    # Identify Archetypes (Map Clusters to Labels)
    centers = kmeans.cluster_centers_
    print("Cluster Centers (Combat, Collect, Explore):\n", centers)
    
    # Mapping Logic: Find the dominant feature for each cluster center
    # index 0 -> pct_combat, 1 -> pct_collect, 2 -> pct_explore
    mapping = {}
    labels = ['Combat', 'Collection', 'Exploration']
    
    for i, center in enumerate(centers):
        dominant_idx = np.argmax(center)
        dominant_label = labels[dominant_idx]
        # Check if multiple clusters claim the same label (rare with normalized distinct features)
        # For now simple mapping. Improving robustness if needed.
        mapping[i] = dominant_label
        
    print("\nCluster Mapping:", mapping)
    
    df['archetype'] = df['cluster'].map(mapping)
    
    # Per-Player Archetype Percentages
    print("\nComputing Per-Player Profiles...")
    # One-hot encode archetypes
    dummies = pd.get_dummies(df['archetype'])
    df_merged = pd.concat([df['userId'], dummies], axis=1)
    
    # Group by User and take mean (since 0/1, mean represents percentage)
    player_profile = df_merged.groupby('userId').mean().reset_index()
    
    # Rename cols for clarity
    rename_map = {}
    for label in labels:
        if label in player_profile.columns:
            rename_map[label] = f'{label}_Pct'
    player_profile.rename(columns=rename_map, inplace=True)
    
    # Fill missing columns with 0 if an archetype wasn't found at all
    for label in labels:
        col_name = f'{label}_Pct'
        if col_name not in player_profile.columns:
             player_profile[col_name] = 0.0
             
    print("\n--- Sample Player Profiles ---")
    print(player_profile.head())
    
    # Export
    df.to_csv(OUT_CLUSTERED, index=False)
    player_profile.to_csv(OUT_ARCHETYPES, index=False)
    print(f"\nSaved clustered data to {OUT_CLUSTERED}")
    print(f"Saved player archetypes to {OUT_ARCHETYPES}")
else:
    print(f"Error: {INPUT_PATH} not found.")